# Referral Bypass Review - Controlled Attribution Analysis

## Executive Summary

- The booking-first route is the primary bypass review because it starts from completed bookings and only accepts referrals submitted before the booking.
- After the test/demo filter, 12 bookings have an eligible prior referral: 8 are honored under the strict booking-source rule and 4 remain possible bypass records.
- The 4 records are **Tier B manual-review cases** because every case has multiple possible prior referrers and one also has a timeline ambiguity.
- The previous 28 same-enquiry findings all have booking dates before referral submission. They are classified as data-integrity records, not bypass cases.
- Agreement value is retained as transaction context only. The extracts do not prove commission eligibility, commission amount, or payout status.

## 1. Context & Decision Rules

**Business question:** Did a referred lead later make a booking without the booking being attributed to `REFERRAL`?

**Primary grain:** one completed booking.

**Primary identity rule:** exact `client + lead_id`.

**Time rule:** `referral_submitted_on <= bookingDate`; `booking_created_on` is used only when `bookingDate` is missing.

**Multiple referrals:** use the latest eligible referral before the booking, while retaining counts of all prior referrals and referrers for ambiguity review.

**Project rule:** a different project does not remove a case from scope, following the dataset guide.

### Honored Definition

A booking is honored only when `booking_source = REFERRAL`. This strict rule is used by both the booking-first route and the same-enquiry route. It checks recorded attribution; it does not prove that commission was paid.

### Confidence Tiers

| Tier | Meaning |
|---|---|
| Tier A | Time-valid exact lead match with one clear prior referrer and no timeline ambiguity |
| Tier B | Time-valid exact lead match, but multiple prior referrers or a timeline ambiguity requires manual review |
| Tier C | Secondary exact encrypted mobile/email match; excluded from the primary count until identity is confirmed |
| Data Integrity | Reverse-time or contradictory records; not treated as bypass evidence |

## 2. Setup & Path Configuration

The notebook runs from either the repository root or the `code` folder. Raw data remains unchanged; outputs are written to `data/derived`.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 100)

In [2]:
# 定位项目目录
# -> 支持从项目根目录或 code 文件夹运行
# -> 原始数据只读，结果写入 data/derived
CURRENT_DIR = Path.cwd().resolve()
ROOT_CANDIDATES = [CURRENT_DIR, CURRENT_DIR.parent, *list(CURRENT_DIR.parents)[:3]]
PROJECT_ROOT = next((path for path in ROOT_CANDIDATES if (path / "data" / "raw").exists()), None)

if PROJECT_ROOT is None:
    searched = "\n".join(str(path / "data" / "raw") for path in ROOT_CANDIDATES)
    raise FileNotFoundError(
        f"Cannot locate data/raw. Current directory: {CURRENT_DIR}\nSearched locations:\n{searched}"
    )

SOURCE_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "derived"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(SOURCE_DIR.glob("*.csv"))
if len(csv_files) < 2:
    raise FileNotFoundError(f"Expected two source CSV files in {SOURCE_DIR}, found {len(csv_files)}")

print(f"Project root folder: {PROJECT_ROOT.name}")
print(f"Source directory: {SOURCE_DIR.relative_to(PROJECT_ROOT)}")
print(f"Output directory: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")

Project root folder: Terra Softech_2
Source directory: data/raw
Output directory: data/derived


## 3. Load and Validate the Two Extracts

IDs are read as pandas strings. Dates and agreement value are converted once after file-role detection.

In [3]:
# 读取原始表
# -> 全部 ID 先按字符串读取
# -> 根据关键字段识别 referrals 和 bookings
loaded_tables = {
    file.name: pd.read_csv(file, dtype="string", low_memory=False)
    for file in csv_files
}

referral_file = next(
    (name for name, table in loaded_tables.items() if {"referral_enquiry_id", "referrer_user_id"}.issubset(table.columns)),
    None,
)
booking_file = next(
    (name for name, table in loaded_tables.items() if {"booking_id", "bookingDate", "booking_source"}.issubset(table.columns)),
    None,
)

if referral_file is None or booking_file is None:
    raise ValueError("Could not identify the referral and booking extracts from their columns.")

referrals_raw = loaded_tables[referral_file]
bookings_raw = loaded_tables[booking_file]

print(f"Referral file: {referral_file} -> {len(referrals_raw):,} rows")
print(f"Booking file: {booking_file} -> {len(bookings_raw):,} rows")

Referral file: referral_submissions.csv -> 5,448 rows
Booking file: booking_extracts.csv -> 5,275 rows


In [4]:
# 校验必要字段
# -> 缺少关键字段时立即停止
# -> 防止后续匹配静默产生错误结果
REQUIRED_REFERRAL_COLUMNS = {
    "Client name", "referral_enquiry_id", "referral_submitted_on", "source", "project_id",
    "referrer_user_id", "referrer_name", "lead_id", "lead_name", "mobileNumber", "email",
    "booking_on_referral_enquiry_id", "booking_on_referral_date", "booking_on_referral_source",
}
REQUIRED_BOOKING_COLUMNS = {
    "Client name", "booking_id", "bookingDate", "booking_created_on", "booking_source",
    "agreementValue", "enquiry_id", "enquiry_source", "lead_id", "project_id",
    "lead_name", "mobileNumber", "email",
}

missing_referral_columns = sorted(REQUIRED_REFERRAL_COLUMNS - set(referrals_raw.columns))
missing_booking_columns = sorted(REQUIRED_BOOKING_COLUMNS - set(bookings_raw.columns))
if missing_referral_columns or missing_booking_columns:
    raise ValueError(
        f"Missing required fields. Referrals: {missing_referral_columns}; Bookings: {missing_booking_columns}"
    )


def parse_datetime(series):
    '''兼容混合格式日期，无法解析的值返回 NaT。'''
    try:
        return pd.to_datetime(series, errors="coerce", format="mixed")
    except TypeError:
        return pd.to_datetime(series, errors="coerce")


def clean_id(series):
    '''统一 ID 格式，保留缺失值，不做 float -> int -> string 转换。'''
    return series.astype("string").str.strip().replace({"": pd.NA, "NULL": pd.NA})


def normalize_source(series):
    '''统一来源字段，空值显示为 <NULL>。'''
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .replace({"": "<NULL>", "NULL": "<NULL>"})
        .fillna("<NULL>")
    )


TEST_DEMO_PATTERN = (
    r"\b(?:test|demo|dummy|sample|training|internal)\b"
    r"|(?:^|[\s_-])test[a-z0-9_-]*"
)


def has_test_demo_name(*series_list):
    '''用明确标记词和 test 前缀标记明显测试或演示记录。'''
    combined = series_list[0].fillna("").astype("string")
    for series in series_list[1:]:
        combined = combined.str.cat(series.fillna("").astype("string"), sep=" ")
    return combined.str.contains(TEST_DEMO_PATTERN, case=False, regex=True, na=False)

In [5]:
# 标准化字段
# -> 日期只转换一次
# -> honored 只使用 booking_source 的 strict 定义
# -> 标记明显 test/demo 姓名
referrals = referrals_raw.assign(
    client_name=clean_id(referrals_raw["Client name"]),
    referral_enquiry_id=clean_id(referrals_raw["referral_enquiry_id"]),
    referral_submitted_at=parse_datetime(referrals_raw["referral_submitted_on"]),
    referral_source_norm=normalize_source(referrals_raw["source"]),
    referral_project_id=clean_id(referrals_raw["project_id"]),
    referral_lead_id=clean_id(referrals_raw["lead_id"]),
    referral_mobile_key=clean_id(referrals_raw["mobileNumber"]),
    referral_email_key=clean_id(referrals_raw["email"]),
    booking_on_referral_id=clean_id(referrals_raw["booking_on_referral_enquiry_id"]),
    booking_on_referral_at=parse_datetime(referrals_raw["booking_on_referral_date"]),
    booking_on_referral_source_norm=normalize_source(referrals_raw["booking_on_referral_source"]),
    referral_test_demo_flag=has_test_demo_name(referrals_raw["lead_name"], referrals_raw["referrer_name"]),
)

bookings = bookings_raw.assign(
    client_name=clean_id(bookings_raw["Client name"]),
    booking_id=clean_id(bookings_raw["booking_id"]),
    enquiry_id=clean_id(bookings_raw["enquiry_id"]),
    booking_date_at=parse_datetime(bookings_raw["bookingDate"]),
    booking_created_at=parse_datetime(bookings_raw["booking_created_on"]),
    booking_source_norm=normalize_source(bookings_raw["booking_source"]),
    enquiry_source_norm=normalize_source(bookings_raw["enquiry_source"]),
    booking_project_id=clean_id(bookings_raw["project_id"]),
    booking_lead_id=clean_id(bookings_raw["lead_id"]),
    booking_mobile_key=clean_id(bookings_raw["mobileNumber"]),
    booking_email_key=clean_id(bookings_raw["email"]),
    agreement_value=pd.to_numeric(bookings_raw["agreementValue"], errors="coerce"),
    booking_test_demo_flag=has_test_demo_name(bookings_raw["lead_name"]),
)

bookings = bookings.assign(
    booking_timeline_at=bookings["booking_date_at"].fillna(bookings["booking_created_at"]),
    honored_strict=bookings["booking_source_norm"].eq("REFERRAL"),
)

if referrals["referral_enquiry_id"].duplicated().any():
    raise ValueError("referral_enquiry_id is not unique at the expected referral-enquiry grain.")
if bookings["booking_id"].duplicated().any():
    raise ValueError("booking_id is not unique at the expected booking grain.")

data_quality_summary = pd.DataFrame([
    {"check": "Referral rows", "value": len(referrals), "status": "Info"},
    {"check": "Booking rows", "value": len(bookings), "status": "Info"},
    {"check": "Duplicate referral enquiry IDs", "value": int(referrals["referral_enquiry_id"].duplicated().sum()), "status": "Pass"},
    {"check": "Duplicate booking IDs", "value": int(bookings["booking_id"].duplicated().sum()), "status": "Pass"},
    {"check": "Referral rows with parsed submitted time", "value": int(referrals["referral_submitted_at"].notna().sum()), "status": "Info"},
    {"check": "Bookings with usable timeline", "value": int(bookings["booking_timeline_at"].notna().sum()), "status": "Info"},
    {"check": "Referral test/demo rows", "value": int(referrals["referral_test_demo_flag"].sum()), "status": "Filtered"},
    {"check": "Booking test/demo rows", "value": int(bookings["booking_test_demo_flag"].sum()), "status": "Filtered"},
])

display(data_quality_summary)

,check,value,status
0,Referral rows,5448,Info
1,Booking rows,5275,Info
2,Duplicate referral enquiry IDs,0,Pass
3,Duplicate booking IDs,0,Pass
4,Referral rows with parsed submitted time,5448,Info
5,Bookings with usable timeline,5275,Info
6,Referral test/demo rows,863,Filtered
7,Booking test/demo rows,562,Filtered


## 4. Data Controls Before Attribution

### Test / Demo Filter

Records whose lead or referrer name contains a clear test/demo marker are excluded from the primary result. The rule covers the full words `test`, `demo`, `dummy`, `sample`, `training`, and `internal`, plus names beginning with `test` such as `TESTKYC` or `Testlead`. The before/after counts remain visible for audit.

### Same-Enquiry Time Gate

The referral extract also contains booking fields for the same enquiry. These rows are checked for chronology before any source comparison. A booking dated before its referral submission is a data-integrity issue and cannot support a bypass claim.

In [6]:
# 过滤明显测试记录
# -> 保留过滤前后数量
# -> 不修改原始 DataFrame
filtered_referrals = referrals.loc[~referrals["referral_test_demo_flag"]]
filtered_bookings = bookings.loc[~bookings["booking_test_demo_flag"]]

filter_base_summary = pd.DataFrame([
    {
        "dataset": "Referral submissions",
        "before_filter": len(referrals),
        "test_demo_removed": int(referrals["referral_test_demo_flag"].sum()),
        "after_filter": len(filtered_referrals),
    },
    {
        "dataset": "Bookings",
        "before_filter": len(bookings),
        "test_demo_removed": int(bookings["booking_test_demo_flag"].sum()),
        "after_filter": len(filtered_bookings),
    },
])

display(filter_base_summary)

,dataset,before_filter,test_demo_removed,after_filter
0,Referral submissions,5448,863,4585
1,Bookings,5275,562,4713


In [7]:
# 检查 same-enquiry 时间顺序
# -> 先判断 booking 是否发生在 referral 之后
# -> reverse-time 只进入数据完整性列表
same_enquiry = referrals.loc[referrals["booking_on_referral_id"].notna()].assign(
    same_enquiry_time_status=lambda df: np.select(
        [
            df["referral_submitted_at"].isna() | df["booking_on_referral_at"].isna(),
            df["booking_on_referral_at"].lt(df["referral_submitted_at"]),
        ],
        ["Missing Date", "Reverse Time"],
        default="Time Valid",
    ),
    same_enquiry_honored_strict=lambda df: df["booking_on_referral_source_norm"].eq("REFERRAL"),
    source_disagreement_flag=lambda df: df["referral_source_norm"].eq("REFERRAL") & ~df["booking_on_referral_source_norm"].eq("REFERRAL"),
)

same_enquiry_time_quality_summary = (
    same_enquiry
    .groupby(["same_enquiry_time_status", "booking_on_referral_source_norm"], dropna=False)
    .agg(
        referral_enquiry_rows=("referral_enquiry_id", "size"),
        unique_leads=("referral_lead_id", "nunique"),
        test_demo_rows=("referral_test_demo_flag", "sum"),
    )
    .reset_index()
    .sort_values(["same_enquiry_time_status", "booking_on_referral_source_norm"])
)

reverse_time_data_integrity_cases = same_enquiry.loc[
    same_enquiry["same_enquiry_time_status"].eq("Reverse Time")
    & same_enquiry["source_disagreement_flag"]
].assign(
    review_class="Data Integrity - Reverse Time",
    issue_reason="Booking date is earlier than referral submission; source mismatch cannot be scored as bypass.",
)

same_enquiry_time_valid_cases = same_enquiry.loc[
    same_enquiry["same_enquiry_time_status"].eq("Time Valid")
].assign(
    review_class=lambda df: np.where(
        df["same_enquiry_honored_strict"],
        "Time Valid - Strictly Honored",
        "Time Valid - Strict Source Review",
    )
)

same_enquiry_export_columns = [
    "review_class", "client_name", "referral_enquiry_id", "referral_submitted_at",
    "referral_source_norm", "referral_lead_id", "lead_name", "referrer_user_id",
    "referrer_name", "referral_project_id", "projectName", "booking_on_referral_id",
    "booking_on_referral_at", "booking_on_referral_source_norm",
    "same_enquiry_time_status", "same_enquiry_honored_strict",
    "source_disagreement_flag", "referral_test_demo_flag",
]

same_enquiry_time_quality_summary.to_csv(OUTPUT_DIR / "same_enquiry_time_quality_summary.csv", index=False)
reverse_time_data_integrity_cases[["issue_reason", *same_enquiry_export_columns]].to_csv(
    OUTPUT_DIR / "reverse_time_data_integrity_cases.csv", index=False
)
same_enquiry_time_valid_cases[same_enquiry_export_columns].to_csv(
    OUTPUT_DIR / "same_enquiry_time_valid_cases.csv", index=False
)

display(same_enquiry_time_quality_summary)
print(f"Reverse-time source-disagreement records: {len(reverse_time_data_integrity_cases):,}")
print(f"Time-valid same-enquiry records: {len(same_enquiry_time_valid_cases):,}")
print(
    "Time-valid records not honored under the strict booking-source definition: "
    f"{int((~same_enquiry_time_valid_cases['same_enquiry_honored_strict']).sum()):,}"
)

,same_enquiry_time_status,booking_on_referral_source_norm,referral_enquiry_rows,unique_leads,test_demo_rows
0,Reverse Time,CHANNEL_PARTNER,28,7,0
1,Reverse Time,REFERRAL,2718,576,544
2,Time Valid,REFERRAL,16,15,8


Reverse-time source-disagreement records: 28
Time-valid same-enquiry records: 16
Time-valid records not honored under the strict booking-source definition: 0


**Interpretation:** The 28 source-disagreement rows are all reverse-time. They are retained for source-data investigation but removed from referral-bypass evidence. Among the 16 time-valid same-enquiry conversions, all are marked `REFERRAL` under the strict booking-source definition.

## 5. Primary Booking-First Attribution Route

This route starts from completed bookings, joins eligible earlier referrals for the same client and exact lead ID, and selects the latest eligible referral. Referral multiplicity is retained to show whether ownership is unambiguous.

In [8]:
# 创建候选匹配对
# -> 只保留同 client、同 lead_id
# -> referral 必须早于或等于 booking
def build_time_valid_pairs(referral_table, booking_table):
    referral_base = referral_table.loc[
        referral_table["referral_lead_id"].notna() & referral_table["referral_submitted_at"].notna(),
        [
            "client_name", "referral_enquiry_id", "referral_submitted_at", "referral_enquiry_status",
            "stage", "referralSource", "referralCode", "sourcePlatform", "referral_project_id",
            "projectName", "referrer_user_id", "referrer_name", "referrer_user_type",
            "referral_lead_id", "lead_name", "lead_hash_id", "referral_mobile_key",
            "referral_email_key", "referral_test_demo_flag",
        ],
    ].rename(
        columns={
            "projectName": "referral_project_name",
            "lead_name": "referral_lead_name",
            "lead_hash_id": "referral_lead_hash_id",
        }
    )

    booking_base = booking_table.loc[
        booking_table["booking_lead_id"].notna() & booking_table["booking_timeline_at"].notna(),
        [
            "client_name", "booking_id", "booking_crm_id", "booking_timeline_at", "booking_date_at",
            "booking_created_at", "bookingStatus", "booking_source_norm", "enquiry_id",
            "enquiry_source_norm", "booking_lead_id", "lead_name", "lead_hash_id",
            "booking_project_id", "projectName", "agreement_value", "honored_strict",
            "booking_test_demo_flag",
        ],
    ].rename(
        columns={
            "lead_name": "booking_lead_name",
            "lead_hash_id": "booking_lead_hash_id",
            "projectName": "booking_project_name",
        }
    )

    pairs = booking_base.merge(
        referral_base,
        left_on=["client_name", "booking_lead_id"],
        right_on=["client_name", "referral_lead_id"],
        how="inner",
        validate="many_to_many",
    )

    return pairs.loc[
        pairs["referral_submitted_at"].le(pairs["booking_timeline_at"])
    ].assign(
        days_between_referral_and_booking=lambda df: (
            df["booking_timeline_at"] - df["referral_submitted_at"]
        ).dt.total_seconds() / 86400,
    )


# 选择最近一次 prior referral
# -> attribution window 只用于敏感性测试
# -> 主结果不自行假设业务窗口
def select_latest_prior_referral(time_valid_pairs, max_window_days=None):
    eligible_pairs = time_valid_pairs
    if max_window_days is not None:
        eligible_pairs = eligible_pairs.loc[
            eligible_pairs["days_between_referral_and_booking"].le(max_window_days)
        ]

    counts = (
        eligible_pairs
        .groupby("booking_id")
        .agg(
            prior_referral_count=("referral_enquiry_id", "nunique"),
            prior_referrer_count=("referrer_user_id", "nunique"),
            prior_project_count=("referral_project_id", "nunique"),
            earliest_prior_referral_at=("referral_submitted_at", "min"),
            latest_prior_referral_at=("referral_submitted_at", "max"),
        )
        .reset_index()
    )

    latest = (
        eligible_pairs
        .sort_values(["booking_id", "referral_submitted_at", "referral_enquiry_id"])
        .drop_duplicates(subset=["booking_id"], keep="last")
        .merge(counts, on="booking_id", how="left", validate="one_to_one")
    )

    return latest.assign(
        same_project=lambda df: df["booking_project_id"].eq(df["referral_project_id"]),
        multiple_referral_flag=lambda df: df["prior_referral_count"].gt(1),
        multiple_referrer_flag=lambda df: df["prior_referrer_count"].gt(1),
        timeline_ambiguity_flag=lambda df: (
            df["booking_created_at"].notna()
            & df["referral_submitted_at"].gt(df["booking_created_at"])
            & df["referral_submitted_at"].le(df["booking_timeline_at"])
        ),
    )

In [9]:
# 生成过滤前后两套结果
# -> 主结果使用过滤后的 exact lead_id 匹配
# -> 过滤前结果仅用于数量对账
unfiltered_pairs = build_time_valid_pairs(referrals, bookings)
filtered_pairs = build_time_valid_pairs(filtered_referrals, filtered_bookings)

unfiltered_matches = select_latest_prior_referral(unfiltered_pairs)
primary_matches = select_latest_prior_referral(filtered_pairs)

primary_matches = primary_matches.assign(
    identity_rule="Exact client + lead_id",
    match_rule="Latest time-valid prior referral; no default attribution window",
    candidate_status=lambda df: np.where(
        df["honored_strict"],
        "Honored under strict rule",
        "Possible bypass - review required",
    ),
    confidence_tier=lambda df: np.select(
        [
            df["honored_strict"],
            ~df["honored_strict"] & ~df["multiple_referrer_flag"] & ~df["timeline_ambiguity_flag"],
            ~df["honored_strict"],
        ],
        [
            "Not a bypass candidate",
            "Tier A - Strong attribution review",
            "Tier B - Ambiguous attribution review",
        ],
        default="Unclassified",
    ),
)

priority_review_cases = primary_matches.loc[~primary_matches["honored_strict"]].sort_values("booking_timeline_at")

priority_columns = [
    "confidence_tier", "candidate_status", "identity_rule", "match_rule",
    "client_name", "booking_id", "booking_timeline_at", "booking_created_at",
    "booking_source_norm", "enquiry_source_norm", "booking_lead_id", "booking_lead_name",
    "booking_project_id", "booking_project_name", "agreement_value",
    "referral_enquiry_id", "referral_submitted_at", "days_between_referral_and_booking",
    "referral_project_id", "referral_project_name", "same_project",
    "referrer_user_id", "referrer_name", "prior_referral_count", "prior_referrer_count",
    "prior_project_count", "multiple_referral_flag", "multiple_referrer_flag",
    "timeline_ambiguity_flag",
]

priority_review_cases = priority_review_cases[priority_columns]
priority_review_cases.to_csv(OUTPUT_DIR / "booking_priority_review_cases.csv", index=False)

print(f"Primary bookings with an eligible prior referral: {primary_matches['booking_id'].nunique():,}")
print(f"Honored under strict rule: {primary_matches.loc[primary_matches['honored_strict'], 'booking_id'].nunique():,}")
print(f"Possible bypass records for review: {priority_review_cases['booking_id'].nunique():,}")
display(
    priority_review_cases[[
        "confidence_tier", "booking_id", "booking_source_norm", "enquiry_source_norm",
        "days_between_referral_and_booking", "prior_referral_count", "prior_referrer_count",
        "timeline_ambiguity_flag",
    ]]
)

Primary bookings with an eligible prior referral: 12
Honored under strict rule: 8
Possible bypass records for review: 4


,confidence_tier,booking_id,booking_source_norm,enquiry_source_norm,days_between_referral_and_booking,prior_referral_count,prior_referrer_count,timeline_ambiguity_flag
0,Tier B - Ambiguous attribution review,1206,<NULL>,CHANNEL_PARTNER,2.250796,10,10,False
11,Tier B - Ambiguous attribution review,935,CHANNEL_PARTNER,CHANNEL_PARTNER,2.356231,61,61,True
8,Tier B - Ambiguous attribution review,3331,CHANNEL_PARTNER,CHANNEL_PARTNER,1.364091,134,134,False
10,Tier B - Ambiguous attribution review,4913,CHANNEL_PARTNER,CHANNEL_PARTNER,3.470859,236,236,False


**Interpretation:** Four possible booking-level bypass records remain after filtering. None qualifies as Tier A because all four have multiple prior referrers; one also has a timing ambiguity between system creation time and the preferred business booking date. They are priority review records, not confirmed commission bypasses.

## 6. Filter and Attribution-Window Sensitivity

This section checks whether the result changes under the test/demo filter or common 30/60/90/180-day attribution windows. The strict booking-source definition remains fixed. These windows are sensitivity checks, not assumed business policy.

In [10]:
# 汇总过滤前后影响
# -> 重点检查过滤是否改变 possible bypass 数量
filter_impact_summary = pd.DataFrame([
    {
        "metric": "Referral rows",
        "before_filter": len(referrals),
        "after_filter": len(filtered_referrals),
    },
    {
        "metric": "Booking rows",
        "before_filter": len(bookings),
        "after_filter": len(filtered_bookings),
    },
    {
        "metric": "Bookings with eligible prior referral",
        "before_filter": unfiltered_matches["booking_id"].nunique(),
        "after_filter": primary_matches["booking_id"].nunique(),
    },
    {
        "metric": "Honored under strict rule",
        "before_filter": unfiltered_matches.loc[unfiltered_matches["honored_strict"], "booking_id"].nunique(),
        "after_filter": primary_matches.loc[primary_matches["honored_strict"], "booking_id"].nunique(),
    },
    {
        "metric": "Possible bypass records",
        "before_filter": unfiltered_matches.loc[~unfiltered_matches["honored_strict"], "booking_id"].nunique(),
        "after_filter": primary_matches.loc[~primary_matches["honored_strict"], "booking_id"].nunique(),
    },
]).assign(
    removed=lambda df: df["before_filter"] - df["after_filter"]
)

filter_impact_summary.to_csv(OUTPUT_DIR / "filter_impact_summary.csv", index=False)
display(filter_impact_summary)

,metric,before_filter,after_filter,removed
0,Referral rows,5448,4585,863
1,Booking rows,5275,4713,562
2,Bookings with eligible prior referral,20,12,8
3,Honored under strict rule,16,8,8
4,Possible bypass records,4,4,0


In [11]:
# 检查 attribution window 敏感性
# -> honored 固定使用 strict booking-source 定义
# -> 不把敏感性窗口写成正式业务规则
sensitivity_rows = []
for cohort_name, pairs in [("Before test/demo filter", unfiltered_pairs), ("After test/demo filter", filtered_pairs)]:
    for window_days in [30, 60, 90, 180, None]:
        window_matches = select_latest_prior_referral(pairs, max_window_days=window_days)
        sensitivity_rows.append({
            "cohort": cohort_name,
            "attribution_window": "No limit" if window_days is None else f"{window_days} days",
            "matched_bookings": window_matches["booking_id"].nunique(),
            "honored_bookings": window_matches.loc[window_matches["honored_strict"], "booking_id"].nunique(),
            "possible_bypass_bookings": window_matches.loc[~window_matches["honored_strict"], "booking_id"].nunique(),
        })

attribution_sensitivity_summary = pd.DataFrame(sensitivity_rows)
attribution_sensitivity_summary.to_csv(OUTPUT_DIR / "attribution_sensitivity_summary.csv", index=False)

display(attribution_sensitivity_summary.loc[
    attribution_sensitivity_summary["cohort"].eq("After test/demo filter")
])

,cohort,attribution_window,matched_bookings,honored_bookings,possible_bypass_bookings
5,After test/demo filter,30 days,11,7,4
6,After test/demo filter,60 days,11,7,4
7,After test/demo filter,90 days,12,8,4
8,After test/demo filter,180 days,12,8,4
9,After test/demo filter,No limit,12,8,4


**Interpretation:** The four primary review records remain under 30-, 60-, 90-, 180-day, and unlimited windows while the strict honored definition stays fixed. This supports stability of the review list. A formal maximum attribution window still requires a business rule.

## 7. Secondary Identity Waterfall

Exact encrypted mobile and email equality is allowed by the dataset guide. These keys are used only after the exact lead-ID route and only within the same client. Any match found only through these fields is Tier C and remains outside the primary count.

In [12]:
# 构建次级身份匹配
# -> 只检查 lead_id 路径没有覆盖的 booking
# -> 手机号和邮箱只做完全相等匹配
def build_secondary_identity_matches(
    referral_table,
    booking_table,
    referral_key,
    booking_key,
    rule_name,
    excluded_booking_ids,
):
    referral_identity = referral_table.loc[
        referral_table[referral_key].notna() & referral_table["referral_submitted_at"].notna(),
        [
            "client_name", referral_key, "referral_enquiry_id", "referral_submitted_at",
            "referral_lead_id", "lead_name", "referrer_user_id", "referrer_name",
            "referral_project_id", "projectName",
        ],
    ].rename(columns={
        referral_key: "identity_key",
        "lead_name": "referral_lead_name",
        "projectName": "referral_project_name",
    })

    booking_identity = booking_table.loc[
        booking_table[booking_key].notna()
        & booking_table["booking_timeline_at"].notna()
        & ~booking_table["booking_id"].isin(excluded_booking_ids),
        [
            "client_name", booking_key, "booking_id", "booking_timeline_at", "booking_source_norm",
            "enquiry_source_norm", "booking_lead_id", "lead_name", "booking_project_id",
            "projectName", "agreement_value", "honored_strict",
        ],
    ].rename(columns={
        booking_key: "identity_key",
        "lead_name": "booking_lead_name",
        "projectName": "booking_project_name",
    })

    candidates = booking_identity.merge(
        referral_identity,
        on=["client_name", "identity_key"],
        how="inner",
        validate="many_to_many",
    )
    candidates = candidates.loc[
        candidates["referral_submitted_at"].le(candidates["booking_timeline_at"])
    ].assign(
        identity_rule=rule_name,
        days_between_referral_and_booking=lambda df: (
            df["booking_timeline_at"] - df["referral_submitted_at"]
        ).dt.total_seconds() / 86400,
    )

    referral_key_cardinality = (
        referral_identity
        .groupby(["client_name", "identity_key"])["referral_lead_id"]
        .nunique(dropna=True)
        .rename("referral_leads_per_identity_key")
        .reset_index()
    )
    booking_key_cardinality = (
        booking_identity
        .groupby(["client_name", "identity_key"])["booking_lead_id"]
        .nunique(dropna=True)
        .rename("booking_leads_per_identity_key")
        .reset_index()
    )

    return (
        candidates
        .sort_values(["booking_id", "referral_submitted_at", "referral_enquiry_id"])
        .drop_duplicates(subset=["booking_id"], keep="last")
        .merge(referral_key_cardinality, on=["client_name", "identity_key"], how="left")
        .merge(booking_key_cardinality, on=["client_name", "identity_key"], how="left")
        .assign(
            identity_collision_flag=lambda df: (
                df["referral_leads_per_identity_key"].gt(1)
                | df["booking_leads_per_identity_key"].gt(1)
            ),
            confidence_tier="Tier C - Secondary identity review",
            candidate_status=lambda df: np.where(
                df["honored_strict"],
                "Honored under strict rule",
                "Possible bypass - identity confirmation required",
            ),
        )
    )


lead_route_booking_ids = set(primary_matches["booking_id"].dropna())
mobile_identity_matches = build_secondary_identity_matches(
    filtered_referrals,
    filtered_bookings,
    "referral_mobile_key",
    "booking_mobile_key",
    "Exact encrypted mobile",
    lead_route_booking_ids,
)
email_identity_matches = build_secondary_identity_matches(
    filtered_referrals,
    filtered_bookings,
    "referral_email_key",
    "booking_email_key",
    "Exact encrypted email",
    lead_route_booking_ids,
)

secondary_identity_all = pd.concat([mobile_identity_matches, email_identity_matches], ignore_index=True)
identity_rule_labels = (
    secondary_identity_all
    .groupby("booking_id")["identity_rule"]
    .agg(lambda values: " + ".join(sorted(set(values))))
    .rename("identity_rules_found")
    .reset_index()
)

identity_match_review_cases = (
    secondary_identity_all
    .sort_values(["booking_id", "referral_submitted_at", "identity_rule"])
    .drop_duplicates(subset=["booking_id"], keep="last")
    .merge(identity_rule_labels, on="booking_id", how="left", validate="one_to_one")
    .sort_values(["candidate_status", "booking_timeline_at"])
)

identity_export_columns = [
    "confidence_tier", "candidate_status", "identity_rules_found", "identity_collision_flag",
    "client_name", "booking_id", "booking_timeline_at", "booking_source_norm",
    "enquiry_source_norm", "booking_lead_id", "booking_lead_name", "booking_project_name",
    "agreement_value", "referral_enquiry_id", "referral_submitted_at", "referral_lead_id",
    "referral_lead_name", "referrer_user_id", "referrer_name", "referral_project_name",
    "days_between_referral_and_booking", "referral_leads_per_identity_key",
    "booking_leads_per_identity_key",
]
identity_match_review_cases = identity_match_review_cases[identity_export_columns]
identity_match_review_cases.to_csv(OUTPUT_DIR / "identity_match_review_cases.csv", index=False)

print(f"Secondary identity review rows: {len(identity_match_review_cases):,}")
print(
    "Secondary identity possible-bypass rows: "
    f"{int(identity_match_review_cases['candidate_status'].str.startswith('Possible bypass').sum()):,}"
)
display(
    identity_match_review_cases[[
        "candidate_status", "identity_rules_found", "identity_collision_flag",
        "booking_id", "booking_source_norm", "enquiry_source_norm",
    ]]
)

Secondary identity review rows: 3
Secondary identity possible-bypass rows: 3


,candidate_status,identity_rules_found,identity_collision_flag,booking_id,booking_source_norm,enquiry_source_norm
0,Possible bypass - identity confirmation required,Exact encrypted mobile,False,2734,CHANNEL_PARTNER,CHANNEL_PARTNER
1,Possible bypass - identity confirmation required,Exact encrypted mobile,True,4849,CHANNEL_PARTNER,CHANNEL_PARTNER
2,Possible bypass - identity confirmation required,Exact encrypted mobile,True,4850,CHANNEL_PARTNER,CHANNEL_PARTNER


**Interpretation:** Secondary encrypted-identity matches are a separate discovery layer. They can recover records missed by `lead_id`, but shared or inconsistent keys can create false matches. Tier C records require identity confirmation before they are added to the primary review list.

## 8. Results and Exported Evidence

The summary below is generated from the same objects used for the case exports. This keeps the notebook, report, and CSV package on one calculation path.

In [13]:
# 生成唯一汇总口径
# -> 报告数字从这张表读取
# -> agreement value 不作为 commission impact
summary_rows = [
    {"metric": "raw_referral_rows", "value": len(referrals), "interpretation": "Raw referral-enquiry records."},
    {"metric": "raw_booking_rows", "value": len(bookings), "interpretation": "Raw non-canceled booking records."},
    {"metric": "test_demo_referral_rows_removed", "value": int(referrals["referral_test_demo_flag"].sum()), "interpretation": "Referral rows excluded by the explicit test/demo name filter."},
    {"metric": "test_demo_booking_rows_removed", "value": int(bookings["booking_test_demo_flag"].sum()), "interpretation": "Booking rows excluded by the explicit test/demo name filter."},
    {"metric": "filtered_referral_rows", "value": len(filtered_referrals), "interpretation": "Referral rows used for primary matching."},
    {"metric": "filtered_booking_rows", "value": len(filtered_bookings), "interpretation": "Booking rows used for primary matching."},
    {"metric": "primary_bookings_with_prior_referral", "value": primary_matches["booking_id"].nunique(), "interpretation": "Filtered bookings with an exact-lead prior referral."},
    {"metric": "primary_honored_strict", "value": primary_matches.loc[primary_matches["honored_strict"], "booking_id"].nunique(), "interpretation": "Primary matches honored under the strict booking-source rule."},
    {"metric": "primary_possible_bypass", "value": priority_review_cases["booking_id"].nunique(), "interpretation": "Primary records requiring bypass review."},
    {"metric": "primary_tier_a", "value": int(priority_review_cases["confidence_tier"].str.startswith("Tier A").sum()), "interpretation": "Unambiguous primary review records."},
    {"metric": "primary_tier_b", "value": int(priority_review_cases["confidence_tier"].str.startswith("Tier B").sum()), "interpretation": "Ambiguous primary review records."},
    {"metric": "same_enquiry_linked_booking_rows", "value": len(same_enquiry), "interpretation": "Referral rows with a same-enquiry booking reference."},
    {"metric": "same_enquiry_reverse_time_rows", "value": int(same_enquiry["same_enquiry_time_status"].eq("Reverse Time").sum()), "interpretation": "Same-enquiry booking date precedes referral submission."},
    {"metric": "reverse_time_source_disagreement_rows", "value": len(reverse_time_data_integrity_cases), "interpretation": "Previous 28 findings reclassified as data integrity."},
    {"metric": "same_enquiry_time_valid_rows", "value": len(same_enquiry_time_valid_cases), "interpretation": "Same-enquiry records eligible for attribution review."},
    {"metric": "same_enquiry_time_valid_strict_unhonored", "value": int((~same_enquiry_time_valid_cases["same_enquiry_honored_strict"]).sum()), "interpretation": "Time-valid same-enquiry records not marked REFERRAL on booking source."},
    {"metric": "secondary_identity_review_rows", "value": len(identity_match_review_cases), "interpretation": "Tier C exact encrypted-identity matches outside primary route."},
    {"metric": "secondary_identity_possible_bypass", "value": int(identity_match_review_cases["candidate_status"].str.startswith("Possible bypass").sum()), "interpretation": "Tier C records requiring identity confirmation."},
]

referral_bypass_summary = pd.DataFrame(summary_rows)

data_quality_summary.to_csv(OUTPUT_DIR / "data_quality_summary.csv", index=False)
referral_bypass_summary.to_csv(OUTPUT_DIR / "referral_bypass_summary.csv", index=False)

display(referral_bypass_summary)

,metric,value,interpretation
0,raw_referral_rows,5448,Raw referral-enquiry records.
1,raw_booking_rows,5275,Raw non-canceled booking records.
2,test_demo_referral_rows_removed,863,Referral rows excluded by the explicit test/demo name filter.
3,test_demo_booking_rows_removed,562,Booking rows excluded by the explicit test/demo name filter.
4,filtered_referral_rows,4585,Referral rows used for primary matching.
5,filtered_booking_rows,4713,Booking rows used for primary matching.
6,primary_bookings_with_prior_referral,12,Filtered bookings with an exact-lead prior referral.
7,primary_honored_strict,8,Primary matches honored under the strict booking-source rule.
8,primary_possible_bypass,4,Primary records requiring bypass review.
9,primary_tier_a,0,Unambiguous primary review records.


## 9. Validation Checks

The notebook fails immediately if a key rule is broken. These checks cover chronology, uniqueness, filtering, source definitions, and output reconciliation.

In [14]:
# 自动校验正式结果
# -> 任一规则失败时停止执行
# -> 防止旧口径或错误时间线进入报告
assert priority_review_cases["booking_id"].is_unique, "Primary review list must be one row per booking."
assert priority_review_cases["referral_submitted_at"].le(priority_review_cases["booking_timeline_at"]).all(), "Primary cases must pass the time-order gate."
assert not priority_review_cases["booking_id"].isin(
    bookings.loc[bookings["booking_test_demo_flag"], "booking_id"]
).any(), "Primary cases must exclude booking test/demo records."
assert len(reverse_time_data_integrity_cases) == int(
    reverse_time_data_integrity_cases["booking_on_referral_at"].lt(
        reverse_time_data_integrity_cases["referral_submitted_at"]
    ).sum()
), "Every data-integrity case must be reverse-time."
assert int((~same_enquiry_time_valid_cases["same_enquiry_honored_strict"]).sum()) == 0, "Time-valid same-enquiry records should reconcile to strict honored status."
assert filter_impact_summary["before_filter"].sub(filter_impact_summary["after_filter"]).eq(
    filter_impact_summary["removed"]
).all(), "Filter counts must reconcile."
assert set(priority_review_cases["confidence_tier"]).issubset({
    "Tier A - Strong attribution review",
    "Tier B - Ambiguous attribution review",
}), "Primary candidates must use the written confidence tiers."

primary_window_check = attribution_sensitivity_summary.loc[
    attribution_sensitivity_summary["cohort"].eq("After test/demo filter")
    & attribution_sensitivity_summary["attribution_window"].isin(["30 days", "60 days", "90 days", "180 days"]),
    "possible_bypass_bookings",
]
assert primary_window_check.nunique() == 1, "Primary candidate count should be stable across tested windows."

expected_output_files = [
    "referral_bypass_summary.csv",
    "data_quality_summary.csv",
    "filter_impact_summary.csv",
    "booking_priority_review_cases.csv",
    "same_enquiry_time_quality_summary.csv",
    "reverse_time_data_integrity_cases.csv",
    "same_enquiry_time_valid_cases.csv",
    "attribution_sensitivity_summary.csv",
    "identity_match_review_cases.csv",
]
missing_outputs = [name for name in expected_output_files if not (OUTPUT_DIR / name).exists()]
assert not missing_outputs, f"Missing output files: {missing_outputs}"

validation_results = pd.DataFrame([
    {"validation": "Primary chronology", "status": "PASS"},
    {"validation": "One row per primary booking", "status": "PASS"},
    {"validation": "Test/demo exclusion", "status": "PASS"},
    {"validation": "Same-enquiry time gate", "status": "PASS"},
    {"validation": "Confidence tiers", "status": "PASS"},
    {"validation": "Window sensitivity", "status": "PASS"},
    {"validation": "Required outputs", "status": "PASS"},
])

display(validation_results)
print("All validation checks passed.")

,validation,status
0,Primary chronology,PASS
1,One row per primary booking,PASS
2,Test/demo exclusion,PASS
3,Same-enquiry time gate,PASS
4,Confidence tiers,PASS
5,Window sensitivity,PASS
6,Required outputs,PASS


All validation checks passed.


## 10. Takeaways

1. **Primary review list:** 4 booking-level records remain after test/demo filtering, but all are Tier B because referral ownership is ambiguous.
2. **Corrected same-enquiry result:** the earlier 28 records are reverse-time data-integrity issues; no time-valid same-enquiry source anomaly remains.
3. **Single honored rule:** both routes use `booking_source = REFERRAL`.
4. **Robustness:** the primary count is unchanged under 30-180-day attribution-window checks.
5. **Secondary discovery:** exact encrypted mobile/email matching produces Tier C records that require identity confirmation and remain outside the primary count.
6. **Business boundary:** the data supports attribution review, not proof of commission avoidance or a commission-impact estimate.